# 核函数与生产率曲线

比较 ETAS 与 ST-NKF 的时间 PDF、空间 PDF 和生产率 κ(m)。时间与空间 PDF 使用双对数坐标，κ 图使用对数纵轴。ETAS 空间核还依赖震级，因此显示为散点。κ 文件名与训练脚本保持一致。

改编自 [zhancx-edu/NeuralKernelFunction 的 plot pattern.ipynb](https://github.com/zhancx-edu/NeuralKernelFunction/blob/febc15aefc316cf128ea217bbd60321c88fbaab1/plot%20pattern.ipynb)，适配本仓库 PyTorch 输出。

在仓库根目录启动 Jupyter，先安装 `requirements-plotting.txt`。修改下面的参数单元格后，从上到下执行。结果保存到 `fig/`；本 Notebook 不启动训练。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "run_models.py").is_file():
    raise FileNotFoundError("请在 ST-NKF-pytorch 仓库根目录启动 Jupyter / nbconvert。")

In [ ]:
DATASETS = ["SCEDC_20", "SCEDC_25", "SCEDC_30", "ComCat", "SaltonSea", "SanJac", "WHITE"]
RESULT_DIR = ROOT / "results"
FIG_DIR = ROOT / "fig"
FIG_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_STEP = 1000
DPI = 500
SHOW_FIGURES = True
MODEL_TAGS = {"ST-NKF": "neural_neural_neural", "ETAS": "empirical_empirical_empirical"}

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "axes.linewidth": 1})

In [ ]:
def load_curve(dataset, tag, x_name, y_name):
    if SAMPLE_STEP < 1:
        raise ValueError("SAMPLE_STEP 必须是正整数。")
    paths = [RESULT_DIR / f"{dataset}_{tag}_{name}.npy" for name in (x_name, y_name)]
    for path in paths:
        if not path.is_file():
            raise FileNotFoundError(f"缺少 {path}；请先运行对应模型，且不要使用 --no-patterns。")
    x, y = [np.load(path, mmap_mode="r") for path in paths]
    if x.ndim != 1 or y.ndim != 1 or x.shape != y.shape or x.size == 0:
        raise ValueError(f"{dataset}: 核曲线必须是长度相同的非空一维数组。")
    # 在不载入完整大数组的情况下抽样，并保留末点以显示尾部。
    indices = np.unique(np.r_[np.arange(0, x.size, SAMPLE_STEP), x.size - 1])
    xs, ys = np.asarray(x[indices]), np.asarray(y[indices])
    if not (np.isfinite(xs).all() and np.isfinite(ys).all()):
        raise ValueError(f"{dataset}: 曲线包含非有限值，请检查训练输出。")
    return xs, ys

def save_figure(fig, stem, pdf=False):
    fig.savefig(FIG_DIR / (stem + ".png"), dpi=DPI, bbox_inches="tight")
    if pdf:
        fig.savefig(FIG_DIR / (stem + ".pdf"), bbox_inches="tight")
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)
    print("Saved:", FIG_DIR / (stem + ".png"))

def plot_pattern(dataset, kind):
    names = {
        "temporal": ("dis_t_train", "pdf_temporal_train"),
        "spatial": ("dis_xy_train", "pdf_spatial_train"),
        "kappa": ("hist_m_train", "kappa_train"),
    }
    x_name, y_name = names[kind]
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for model, color in (("ETAS", "#1f77b4"), ("ST-NKF", "#d62728")):
        x, y = load_curve(dataset, MODEL_TAGS[model], x_name, y_name)
        mask = (y > 0)
        if kind != "kappa":
            mask &= (x > 0)
        if not mask.any():
            raise ValueError(f"{dataset} {model} {kind}: 没有可在对数坐标显示的正值。")
        if kind == "spatial" and model == "ETAS":
            ax.scatter(x[mask], y[mask], s=2, color=color, label=model)
        else:
            ax.plot(x[mask], y[mask], color=color, lw=1.5, label=model)
    ax.set_yscale("log")
    if kind != "kappa":
        ax.set_xscale("log")
    ax.set_xlabel({"temporal": "Time lag (days)", "spatial": "Distance (km)",
                   "kappa": "Magnitude"}[kind])
    ax.set_ylabel("Expected number of offspring" if kind == "kappa" else "Probability density")
    ax.set_title(dataset)
    ax.grid(True, which="both", ls="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    stem = f"{dataset}-kappa_pattern" if kind == "kappa" else f"{dataset}-{kind}_pattern-loglog"
    save_figure(fig, stem)

In [ ]:
for dataset in DATASETS:
    for kind in ("spatial", "temporal", "kappa"):
        plot_pattern(dataset, kind)